# 02 — Feature Engineering

Loads raw data, computes per-match features, and writes `../data/processed/training_set.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR   = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
matches       = pd.read_csv(DATA_DIR / "matches.csv",       parse_dates=["date"])
fifa_rankings = pd.read_csv(DATA_DIR / "fifa_rankings.csv")
elo_ratings   = pd.read_csv(DATA_DIR / "elo_ratings.csv")
market_values = pd.read_csv(DATA_DIR / "market_values.csv")
injuries      = pd.read_csv(DATA_DIR / "injuries.csv")

# Filter to 2023+ only (exclude 2022)
matches = matches[matches["date"] >= "2023-01-01"].copy()

print(f"matches:       {len(matches):,} rows (2023-2026 only)")
print(f"fifa_rankings: {len(fifa_rankings):,} rows")
print(f"elo_ratings:   {len(elo_ratings):,} rows")
print(f"market_values: {len(market_values):,} rows")
print(f"injuries:      {len(injuries):,} rows")
print(f"\nMatches by year:")
print(matches["date"].dt.year.value_counts().sort_index())

matches:       3,563 rows (2023-2026 only)
fifa_rankings: 211 rows
elo_ratings:   229 rows
market_values: 20 rows
injuries:      19 rows

Matches by year:
date
2023    1054
2024    1231
2025    1002
2026     276
Name: count, dtype: int64


In [3]:
NAME_MAP = {
    # FIFA / AFC variants
    "Korea Republic":        "South Korea",
    "IR Iran":               "Iran",
    "China PR":              "China",
    "Chinese Taipei":        "Taiwan",
    "Hong Kong, China":      "Hong Kong",
    "Kyrgyz Republic":       "Kyrgyzstan",
    "Macau":                 "Macao",
    # CAF variants
    "Congo DR":              "DR Congo",
    "Congo, DR":             "DR Congo",
    # UEFA variants
    "Czechia":               "Czech Republic",
    "North Macedonia":       "North Macedonia",
    # CONCACAF variants
    "USA":                   "United States",
    "US Virgin Islands":     "United States Virgin Islands",
    # Misc
    "Cabo Verde":            "Cape Verde",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Eswatini":              "Swaziland",
    "Timor-Leste":           "East Timor",
}

def normalize_names(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].replace(NAME_MAP)
    return df

matches       = normalize_names(matches,       ["home_team", "away_team"])
fifa_rankings = normalize_names(fifa_rankings, ["team"])
elo_ratings   = normalize_names(elo_ratings,   ["team"])
market_values = normalize_names(market_values, ["team"])

print("Name normalization applied.")

Name normalization applied.


In [4]:
# fifa_lookup: team → {rank, points, confederation}
fifa_lookup = {
    row["team"]: {
        "rank":          row["rank"],
        "points":        row["points"],
        "confederation": row["confederation"],
    }
    for _, row in fifa_rankings.iterrows()
}

# elo_lookup: team → elo_rating
elo_lookup = {
    row["team"]: row["elo_rating"]
    for _, row in elo_ratings.iterrows()
}

# market_lookup: team → squad_total_value_eur
market_lookup = {
    row["team"]: row["squad_total_value_eur"]
    for _, row in market_values.iterrows()
}

print(f"fifa_lookup:   {len(fifa_lookup)} teams")
print(f"elo_lookup:    {len(elo_lookup)} teams")
print(f"market_lookup: {len(market_lookup)} teams")

fifa_lookup:   211 teams
elo_lookup:    229 teams
market_lookup: 20 teams


In [5]:
def compute_rolling_features(matches_df: pd.DataFrame, team: str, n: int = 10) -> dict:
    """Return form stats for *team* across its last N matches in matches_df."""
    team_mask = (matches_df["home_team"] == team) | (matches_df["away_team"] == team)
    team_matches = matches_df[team_mask].tail(n)

    if team_matches.empty:
        return {"form": 0.5, "goals_scored_avg": 1.0, "goals_conceded_avg": 1.0}

    wins, goals_scored, goals_conceded = 0, 0, 0
    for _, m in team_matches.iterrows():
        if m["home_team"] == team:
            gs, gc = m["home_score"], m["away_score"]
        else:
            gs, gc = m["away_score"], m["home_score"]
        goals_scored    += gs
        goals_conceded  += gc
        if gs > gc:
            wins += 1

    n_played = len(team_matches)
    return {
        "form":               wins / n_played,
        "goals_scored_avg":   goals_scored   / n_played,
        "goals_conceded_avg": goals_conceded / n_played,
    }

In [6]:
def compute_h2h(matches_df: pd.DataFrame, home_team: str, away_team: str, n: int = 5) -> float:
    """Return win ratio for home_team in the last N head-to-head meetings."""
    mask = (
        ((matches_df["home_team"] == home_team) & (matches_df["away_team"] == away_team)) |
        ((matches_df["home_team"] == away_team) & (matches_df["away_team"] == home_team))
    )
    h2h_matches = matches_df[mask].tail(n)

    if h2h_matches.empty:
        return 0.5

    wins = 0
    for _, m in h2h_matches.iterrows():
        if m["home_team"] == home_team:
            hs, as_ = m["home_score"], m["away_score"]
        else:
            hs, as_ = m["away_score"], m["home_score"]
        if hs > as_:
            wins += 1

    return wins / len(h2h_matches)

In [7]:
TOURNAMENT_IMPORTANCE = {
    "Friendly":                      0.3,
    "FIFA World Cup qualification":   0.7,
    "UEFA Euro qualification":        0.7,
    "Copa América":                   0.9,
    "UEFA Euro":                      0.9,
    "FIFA World Cup":                 1.0,
    "UEFA Nations League":            0.6,
    # common extras
    "Africa Cup of Nations":          0.9,
    "Africa Cup of Nations qualification": 0.6,
    "AFC Asian Cup":                  0.9,
    "AFC Asian Cup qualification":    0.6,
    "CONCACAF Gold Cup":              0.8,
    "CONCACAF Nations League":        0.6,
    "AFF Championship":               0.5,
}

def get_importance(tournament: str) -> float:
    """Return a numeric importance weight for a tournament string."""
    if tournament in TOURNAMENT_IMPORTANCE:
        return TOURNAMENT_IMPORTANCE[tournament]
    # partial-match fallback
    lower = tournament.lower()
    if "world cup" in lower:
        return 0.9 if "qualif" in lower else 1.0
    if "qualif" in lower:
        return 0.6
    if "friendly" in lower:
        return 0.3
    return 0.5

In [8]:
def get_sample_weight(match_date: pd.Timestamp) -> float:
    """Recency weight: more recent matches matter more. 2026 overweighted."""
    year_weights = {2026: 1.0, 2025: 0.6, 2024: 0.3, 2023: 0.15}
    return year_weights.get(match_date.year, 0.0)  # 2022 and earlier get 0 (excluded)

In [9]:
# Sort matches chronologically so prior-match slicing is correct
matches = matches.sort_values("date").reset_index(drop=True)

DEFAULT_RANK  = 100
DEFAULT_PTS   = 1300
DEFAULT_ELO   = 1500
DEFAULT_CONF  = "UNK"

rows = []

for i, match in matches.iterrows():
    if i % 500 == 0:
        print(f"  processing match {i:,} / {len(matches):,}...")

    home, away = match["home_team"], match["away_team"]

    # --- no-leakage: only matches strictly before this one ---
    prior = matches.iloc[:i]

    # Rolling form
    home_form = compute_rolling_features(prior, home)
    away_form = compute_rolling_features(prior, away)

    # H2H
    h2h_wins = compute_h2h(prior, home, away)

    # Static lookups with defaults
    home_fifa = fifa_lookup.get(home, {})
    away_fifa = fifa_lookup.get(away, {})

    home_rank    = home_fifa.get("rank",          DEFAULT_RANK)
    away_rank    = away_fifa.get("rank",          DEFAULT_RANK)
    home_pts     = home_fifa.get("points",        DEFAULT_PTS)
    away_pts     = away_fifa.get("points",        DEFAULT_PTS)
    home_conf    = home_fifa.get("confederation", DEFAULT_CONF)
    away_conf    = away_fifa.get("confederation", DEFAULT_CONF)

    home_elo     = elo_lookup.get(home, DEFAULT_ELO)
    away_elo     = elo_lookup.get(away, DEFAULT_ELO)

    home_mkt     = market_lookup.get(home, None)
    away_mkt     = market_lookup.get(away, None)
    if home_mkt and away_mkt and away_mkt > 0:
        squad_value_ratio = home_mkt / away_mkt
    else:
        squad_value_ratio = 1.0

    # Result
    hs, as_ = match["home_score"], match["away_score"]
    goal_diff = hs - as_
    if goal_diff > 0:
        result = "W"
    elif goal_diff < 0:
        result = "L"
    else:
        result = "D"

    rows.append({
        "date":                    match["date"],
        "home_team":               home,
        "away_team":               away,
        "tournament":              match["tournament"],
        # ELO / FIFA
        "elo_diff":                home_elo - away_elo,
        "fifa_rank_diff":          away_rank - home_rank,   # positive = home ranked better
        "fifa_points_diff":        home_pts  - away_pts,
        # Market
        "squad_value_ratio":       squad_value_ratio,
        # Form
        "form_home":               home_form["form"],
        "form_away":               away_form["form"],
        "goals_scored_avg_home":   home_form["goals_scored_avg"],
        "goals_scored_avg_away":   away_form["goals_scored_avg"],
        "goals_conceded_avg_home": home_form["goals_conceded_avg"],
        "goals_conceded_avg_away": away_form["goals_conceded_avg"],
        # H2H
        "h2h_wins":                h2h_wins,
        # Context
        "match_importance":        get_importance(match["tournament"]),
        "confederation_home":      home_conf,
        "confederation_away":      away_conf,
        "neutral_venue":           int(match["neutral"]) if pd.notna(match["neutral"]) else 0,
        # Target
        "result":                  result,
        "goal_diff":               goal_diff,
        "sample_weight":           get_sample_weight(match["date"]),
    })

training_df = pd.DataFrame(rows)
print(f"\nBuilt {len(training_df):,} rows")

  processing match 0 / 3,563...


  processing match 500 / 3,563...


  processing match 1,000 / 3,563...


  processing match 1,500 / 3,563...


  processing match 2,000 / 3,563...


  processing match 2,500 / 3,563...


  processing match 3,000 / 3,563...


  processing match 3,500 / 3,563...



Built 3,563 rows


In [10]:
out_path = OUTPUT_DIR / "training_set.csv"
training_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Shape: {training_df.shape}")
print("\nResult distribution:")
print(training_df["result"].value_counts())
print("\nNumerical summary:")
print(training_df.describe())

Saved → ../data/processed/training_set.csv
Shape: (3563, 22)

Result distribution:
result
W    1647
L    1044
D     872
Name: count, dtype: int64

Numerical summary:
                                date     elo_diff  fifa_rank_diff  \
count                           3563  3563.000000      3563.00000   
mean   2024-09-10 15:52:59.680044800    36.129947         6.88970   
min              2023-01-02 00:00:00 -1080.000000      -202.00000   
25%              2023-11-16 00:00:00  -164.000000       -29.00000   
50%              2024-09-08 00:00:00    37.000000         7.00000   
75%              2025-06-17 12:00:00   249.000000        44.00000   
max              2026-06-27 00:00:00  1133.000000       202.00000   
std                              NaN   319.720725        58.76098   

       fifa_points_diff  squad_value_ratio    form_home    form_away  \
count       3563.000000        3563.000000  3563.000000  3563.000000   
mean          34.746600           1.017213     0.407396     0.388438